A better version of `power_shendure_vs_minp.ipynb` with a more representative distribution of positive and negative effects. Specifically, we will be using the real values, plus many negatives.

Based on UKBB paper, expect ~30% of library to be active. So we will add 2x original size of negatives... 

We will also reduce computational burden by producing half-orthos.

# Imports & dask cluster creation

In [25]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
from pathlib import Path

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="32G",#memory per slurm job
        processes=1,#dask workers per slurm jobTrueT
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=4:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=2)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

2026-01-23 14:07:51,082 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='a1132u05n01.mghpcc.ycrc.yale.edu:51860', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/tornado/web.py", line 3375, in wrapper
    return method(self, *args, **kwargs)
  File "/home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the app with a larger value for --session-t

In [27]:
client.dashboard_link

'http://127.0.0.1:8787/status'

# Ground truth creation

We will use parameter estimates from by_cell_type models as we expect these to be the most accurate.

In [28]:
data_root=Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")

In [29]:
primordial=scm.ortho.load(client,data_root/"shendure","ortho_primordial_v4")

In [30]:
primordial.compute_model_qc()

In [31]:
import pandas as pd

In [32]:
vals=[]
for key in primordial.by_cell_qc.keys():
    key="SurfaceEctoderm"
    working=primordial.by_cell_qc[key]['dat'].reset_index().drop(columns=["mean(umis_mpra_bc)"])
    working["cell_type"]=key
    vals.append(working)
vals=pd.concat(vals)
vals=vals.groupby(["cre_id","cell_type"]).mean().reset_index()
vals

,cre_id,cell_type,mu
0,Bend5_chr4_8168,SurfaceEctoderm,0.019097
1,Bend5_chr4_8174,SurfaceEctoderm,0.01163
2,Bend5_chr4_8175,SurfaceEctoderm,0.671256
3,Bend5_chr4_8179,SurfaceEctoderm,0.019683
4,Bend5_chr4_8199,SurfaceEctoderm,0.004408
...,...,...,...
134,Txndc12_chr4_7978,SurfaceEctoderm,0.153156
135,eef1aP,SurfaceEctoderm,88.727592
136,pgk1P,SurfaceEctoderm,16.479346
137,reference,SurfaceEctoderm,0.007241


Now that we have reasonable mu estimates for the real CRE, let us add 200% "indistinguishable from minP".

In [33]:
#make "corresponding" inactive CREs...
mapping = {
    val: f"inactive_{i}"
    for i, val in enumerate(vals["cre_id"].unique())
}
mapping

{'Bend5_chr4_8168': 'inactive_0',
 'Bend5_chr4_8174': 'inactive_1',
 'Bend5_chr4_8175': 'inactive_2',
 'Bend5_chr4_8179': 'inactive_3',
 'Bend5_chr4_8199': 'inactive_4',
 'Btg1_chr10_9578': 'inactive_5',
 'Btg1_chr10_9588': 'inactive_6',
 'Btg1_chr10_9593': 'inactive_7',
 'Btg1_chr10_9612': 'inactive_8',
 'Btg1_chr10_9613': 'inactive_9',
 'Cdk5r1_chr11_12559': 'inactive_10',
 'Cdk5r1_chr11_12562': 'inactive_11',
 'Cdk5r1_chr11_12574': 'inactive_12',
 'Cdk5r1_chr11_12575': 'inactive_13',
 'Cdk5r1_chr11_12582': 'inactive_14',
 'Cdk5r1_chr11_12590': 'inactive_15',
 'Cited2_chr10_1253': 'inactive_16',
 'Cited2_chr10_1254': 'inactive_17',
 'Cited2_chr10_1267': 'inactive_18',
 'Col1a1_chr11_15258': 'inactive_19',
 'Col1a1_chr11_15259': 'inactive_20',
 'Col1a1_chr11_15270': 'inactive_21',
 'Col1a1_chr11_15275': 'inactive_22',
 'Col1a1_chr11_15276': 'inactive_23',
 'Col1a1_chr11_15301': 'inactive_24',
 'Col1a1_chr11_15307': 'inactive_25',
 'Col1a1_chr11_15316': 'inactive_26',
 'Col1a1_chr11_15

In [34]:
minP=scm.SHENDURE_BOUNDS.reference_activity
inactive=vals.copy().drop(columns=["mu"])
inactive["cre_id"] = inactive["cre_id"].map(mapping)
inactive["mu"]=minP
inactive

,cre_id,cell_type,mu
0,inactive_0,SurfaceEctoderm,0.019311
1,inactive_1,SurfaceEctoderm,0.019311
2,inactive_2,SurfaceEctoderm,0.019311
3,inactive_3,SurfaceEctoderm,0.019311
4,inactive_4,SurfaceEctoderm,0.019311
...,...,...,...
134,inactive_134,SurfaceEctoderm,0.019311
135,inactive_135,SurfaceEctoderm,0.019311
136,inactive_136,SurfaceEctoderm,0.019311
137,inactive_137,SurfaceEctoderm,0.019311


This is 100%. Let us double to 200%...

In [35]:
# duplicated version with _b appended
inactive_b = inactive.copy()
inactive_b["cre_id"] = inactive_b["cre_id"] + "_b"

# stack them
inactive_double = pd.concat([inactive, inactive_b], ignore_index=True)
inactive_double

,cre_id,cell_type,mu
0,inactive_0,SurfaceEctoderm,0.019311
1,inactive_1,SurfaceEctoderm,0.019311
2,inactive_2,SurfaceEctoderm,0.019311
3,inactive_3,SurfaceEctoderm,0.019311
4,inactive_4,SurfaceEctoderm,0.019311
...,...,...,...
273,inactive_134_b,SurfaceEctoderm,0.019311
274,inactive_135_b,SurfaceEctoderm,0.019311
275,inactive_136_b,SurfaceEctoderm,0.019311
276,inactive_137_b,SurfaceEctoderm,0.019311


Then stack with original gt...

In [36]:
final_gt=pd.concat([vals,inactive_double],ignore_index=True).rename({"mu":"true_mean"},axis=1)
final_gt

,cre_id,cell_type,true_mean
0,Bend5_chr4_8168,SurfaceEctoderm,0.019097
1,Bend5_chr4_8174,SurfaceEctoderm,0.01163
2,Bend5_chr4_8175,SurfaceEctoderm,0.671256
3,Bend5_chr4_8179,SurfaceEctoderm,0.019683
4,Bend5_chr4_8199,SurfaceEctoderm,0.004408
...,...,...,...
412,inactive_134_b,SurfaceEctoderm,0.019311
413,inactive_135_b,SurfaceEctoderm,0.019311
414,inactive_136_b,SurfaceEctoderm,0.019311
415,inactive_137_b,SurfaceEctoderm,0.019311


In [37]:
assert len(final_gt[["cre_id","cell_type"]].drop_duplicates()) == len(final_gt)

# Creating artificial libraries

In [38]:
libraries=[scm.simulate_library(CREs=final_gt["cre_id"],
                 library_model=scm.SHENDURE_BOUNDS.library_model)
                 for i in range(5)]

In [39]:
libraries[2]

,cre_id,mpra_bc,abundance
0,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAAAA,0.000020
1,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAAAC,0.000015
2,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAAAG,0.000011
3,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAAAT,0.000010
4,Bend5_chr4_8168,AAAAAAAAAAAAAAAAAACA,0.000017
...,...,...,...
55128,inactive_138_b,AAAAAAAAAAAATCCTCCGA,0.000015
55129,inactive_138_b,AAAAAAAAAAAATCCTCCGC,0.000035
55130,inactive_138_b,AAAAAAAAAAAATCCTCCGG,0.000024
55131,inactive_138_b,AAAAAAAAAAAATCCTCCGT,0.000013


In [40]:
final_gt.dtypes

cre_id                     object
cell_type                  object
true_mean    Sparse[float64, 1.0]
dtype: object

# Creating sim

In [41]:
sim=scm.de_novo_simulation(location=data_root,
                            name="twothird_pow_sim_2026-01-22",
                            client=client,
                            libraries=libraries,
                            library_mapping="corresponding",
                            n_sims=5,
                            experiment_bounds=scm.SHENDURE_BOUNDS,
                            ground_truth=final_gt)

scMPRAforge: INFO: 'state.parquet' found for 'twothird_pow_sim_2026-01-22', loading.


In [42]:
sim.gamut()

In [43]:
#sim.save()

In [44]:
#sim

Make the hypotheses...

In [45]:
#spread_hypothesis.to_tsv(f"{data_root}/pow_sim_2026-01-03_hypo.tsv")
#hs_all_cre = scm.make_all_by_cre_hypotheses(
#    counts=demo_counts,
#    reference_cell_type="reference",
#)

In [46]:
sim.ground_truth.dtypes

cre_id        object
cell_type     object
true_mean    float64
dtype: object

In [47]:
#client.close()
#cluster.close()

scMPRAforge: INFO: 1 cells_df has col Index(['cell_type', 'cell_bc'], dtype='object') drawn_library had cols Index(['Unnamed: 0', 'cre_id', 'mpra_bc', 'abundance'], dtype='object')
scMPRAforge: INFO: 1 cells_df has col Index(['cell_type', 'cell_bc'], dtype='object') drawn_library had cols Index(['Unnamed: 0', 'cre_id', 'mpra_bc', 'abundance'], dtype='object')
scMPRAforge: INFO: 1 cells_df has col Index(['cell_type', 'cell_bc'], dtype='object') drawn_library had cols Index(['Unnamed: 0', 'cre_id', 'mpra_bc', 'abundance'], dtype='object')
scMPRAforge: INFO: 1 cells_df has col Index(['cell_type', 'cell_bc'], dtype='object') drawn_library had cols Index(['Unnamed: 0', 'cre_id', 'mpra_bc', 'abundance'], dtype='object')
scMPRAforge: INFO: 2 1389000
scMPRAforge: INFO: 2 1381118
scMPRAforge: INFO: 2 1381570
scMPRAforge: INFO: 2 1384698
scMPRAforge: INFO: [debug] dumped df to: debug_df_4ea7805e79e3475eb9d8b24187b360b1.tsv.gz
scMPRAforge: INFO: [debug] dumped df to: debug_df_23eed58c35b04a6baf3d